# Kaggle 训练阶段 1：第 5 轮检查点

本 Notebook 固定训练上限为 15 epoch，并在第 5 个完整 epoch 边界触发受控 PREEMPTED。引擎会保存并推送 latest checkpoint、pause-manifest 和服务审计；退出码 75 是预期结果。

In [ ]:
import os

# 21e4c423 包含当前 CLI、project-dir、自动预检和恢复契约；可在发布新版本后覆盖。
os.environ.setdefault('DL_HELPER_GIT_REPO', 'https://github.com/lhiqwj173/dl_helper.git')
os.environ.setdefault('DL_HELPER_GIT_REF', 'master')
os.environ.setdefault('DL_HELPER_MNIST_PATH', '/kaggle/input/datasets/vikramtiwari/mnist-numpy/mnist.npz')
os.environ.setdefault('DL_HELPER_RUN_ID', 'mnist-15epoch-stage1')
os.environ.setdefault('ALIST_HOST', 'http://139.196.47.52')
os.environ.setdefault('ALIST_BASE_PATH', '/dl-helper/mnist-15epoch')
os.environ.setdefault('WECOM_TO_USER', '@all')
print('训练 run-id:', os.environ['DL_HELPER_RUN_ID'])
print('代码 revision:', os.environ['DL_HELPER_GIT_REF'])

In [ ]:
import os, subprocess, sys

repo_dir = '/kaggle/working/dl-helper'
if os.path.exists(repo_dir):
    raise RuntimeError(f'目录已存在，请新建 Kaggle Session 后重试: {repo_dir}')

def checked(argv, *, cwd=None):
    proc = subprocess.run(argv, cwd=cwd, capture_output=True, text=True, encoding='utf-8')
    if proc.stdout:
        print(proc.stdout, end='')
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr, file=sys.stderr, end='')
        raise RuntimeError(f'命令失败，退出码 {proc.returncode}: {argv}')
    return proc

checked(['git', 'clone', os.environ['DL_HELPER_GIT_REPO'], repo_dir])
checked(['git', 'checkout', os.environ['DL_HELPER_GIT_REF']], cwd=repo_dir)
head = checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
expected = checked(['git', 'rev-parse', os.environ['DL_HELPER_GIT_REF']], cwd=repo_dir).stdout.strip()
if head.lower() != expected.lower():
    raise RuntimeError(f'checkout HEAD 不匹配: {head} != {expected}')
os.environ['DL_HELPER_REPO_DIR'] = repo_dir
checked([sys.executable, f'{repo_dir}/envs/kaggle_bootstrap.py'], cwd=repo_dir)
print('[bootstrap] fixed revision:', head)

In [ ]:
from pathlib import Path
import yaml

source = Path('/kaggle/working/dl-helper/examples/configs/kaggle/mnist.yaml')
config_path = Path('/kaggle/working/dl-helper-stage1.yaml')
with source.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
config['experiment']['data_path'] = os.environ['DL_HELPER_MNIST_PATH']
config['training']['max_epochs'] = 15
config['backend']['torch']['mixed_precision'] = 'no'
config['distributed']['num_processes'] = 1
config['selection']['patience'] = 30
config['checkpoint']['every_epochs'] = 5
config['checkpoint']['keep_last'] = 2
config['run']['id'] = os.environ['DL_HELPER_RUN_ID']
config['run']['source_revision'] = os.environ['DL_HELPER_GIT_REF']
config['remote'] = {'type': 'alist', 'host': os.environ['ALIST_HOST'], 'base_path': os.environ['ALIST_BASE_PATH'], 'user_secret_key': 'ALIST_USER', 'password_secret_key': 'ALIST_PWD', 'connect_timeout_seconds': 10, 'read_timeout_seconds': 600, 'max_attempts': 3, 'async_upload': False, 'failure_policy': 'required'}
config['notifications'] = {'type': 'wecom', 'corp_id_secret_key': 'WECOM_CORP_ID', 'corp_secret_key': 'WECOM_CORP_SECRET', 'agent_id_secret_key': 'WECOM_AGENT_ID', 'to_user': os.environ['WECOM_TO_USER'], 'connect_timeout_seconds': 10, 'read_timeout_seconds': 30, 'max_attempts': 3, 'failure_policy': 'required'}
with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
print('配置已写入:', config_path)

In [ ]:
import subprocess, sys

preflight = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(config_path), '--project-dir', '/kaggle/working/dl-helper/examples', '--experiment', 'experiments.mnist:build_experiment', '--preflight-only'], cwd=repo_dir, text=True, encoding='utf-8')
if preflight.returncode != 0:
    raise RuntimeError(f'预检失败，退出码 {preflight.returncode}')
print('preflight exit code:', preflight.returncode)

In [ ]:
# 在当前进程注入“第 5 个完整 epoch 后暂停”的预算策略，保留正常 checkpoint/远端终结流程。
import json
import sys
from dataclasses import replace

repo_path = '/kaggle/working/dl-helper'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from dl_helper.training import platform as platform_module
from dl_helper.training import cli

OriginalBudget = platform_module.RuntimeBudget
class StopAfterFiveEpochs(OriginalBudget):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.completed_epochs = 0

    def complete_epoch(self, started_at):
        forecast = super().complete_epoch(started_at)
        self.completed_epochs += 1
        return replace(forecast, should_preempt=self.completed_epochs == 5)

platform_module.RuntimeBudget = StopAfterFiveEpochs
try:
    exit_code = cli.main(['train', '--config', str(config_path), '--project-dir', '/kaggle/working/dl-helper/examples', '--experiment', 'experiments.mnist:build_experiment', '--resume', 'none', '--run-id', os.environ['DL_HELPER_RUN_ID']])
finally:
    platform_module.RuntimeBudget = OriginalBudget
if exit_code != 75:
    raise RuntimeError(f'阶段 1 必须在第 5 轮以 75 暂停，实际退出码: {exit_code}')
checkpoint_root = Path('/kaggle/working/dl-helper-runs/runs') / os.environ['DL_HELPER_RUN_ID'] / 'checkpoints'
with (checkpoint_root / 'latest.json').open('r', encoding='utf-8') as f:
    latest = json.load(f)
with (checkpoint_root / latest['path'] / 'checkpoint-manifest.json').open('r', encoding='utf-8') as f:
    manifest = json.load(f)
if manifest['epoch'] != 5:
    raise RuntimeError(f"阶段 1 checkpoint epoch 必须为 5，实际为 {manifest['epoch']}")
print('阶段 1 完成：第 5 轮 checkpoint 已保存并推送:', manifest['checkpoint_id'])